In [ ]:
from astropy.table import Table
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import pymc3 as pm
import pymc3_ext as pmx
import exoplanet as xo
import os
from corner import corner
import theano.tensor as tt
from tqdm.notebook import tqdm
from scipy.stats import median_abs_deviation
import pandas as pd
from astroquery.mast import Catalogs
from scipy.interpolate import UnivariateSpline
import pandas as pd
import numpy as np


In [ ]:
def read_parsec_isochrone(path):
    header_line = None
    
    with open(path) as f:
        for line in f:
            if line.startswith("#") and "Gmag" in line:
                header_line = line
                break   # ← aquí sí paramos cuando encontramos el bueno

    columns = header_line.replace("#", "").split()

    iso = pd.read_csv(
        path,
        sep=r"\s+",
        comment="#",
        names=columns,
        engine="python"
    )
    
    iso.columns = iso.columns.str.strip()
    return iso

iso100 = read_parsec_isochrone("/home/luciam/Documents/catalogos_tesis/isocronas-parsec/100Myr.dat")


In [ ]:
print(iso100.columns.tolist())


In [ ]:
def get_star_from_TIC(tic_id):
    cat = Catalogs.query_object(f"TIC {tic_id}", catalog="TIC", radius=0.001)

    plx   = cat['plx'][0]        # mas
    Gmag  = cat['GAIAmag'][0]
    BPmag = cat['gaiabp'][0]
    RPmag = cat['gaiarp'][0]
    Teff = cat['Teff'][0]
    Lum  = cat['lum'][0]

    # Distancia
    d_pc = 1000 / plx

    # Magnitud absoluta
    Mg = Gmag - 5*np.log10(d_pc) + 5

    # Color
    color = BPmag - RPmag

    return color, Mg, Teff, Lum

In [ ]:
tic_id= 125754991
color_star, Mg_star , Teff_star, Lum_star = get_star_from_TIC(tic_id)


In [ ]:
BP = iso100["G_BPbrmag"]
RP = iso100["G_RPmag"]
G  = iso100["Gmag"]
Te_iso = 10**iso100["logTe"]     # Kelvin
L_iso  = 10**iso100["logL"]      # L / Lsun

color_iso = BP - RP
Mg_iso = G


In [ ]:
mask = (L_iso > 1e-8) & (L_iso < 1e2)

Te_iso_clean = Te_iso[mask]
L_iso_clean = L_iso[mask]


In [ ]:

plt.figure(figsize=(7,6))

plt.scatter(Te_iso_clean, L_iso_clean, color='black', label='PARSEC 100 Myr')
plt.scatter(Teff_star, Lum_star, color='red', s=120, zorder=5, label=f"TIC {tic_id}")

plt.xscale('log')
plt.yscale('log')

plt.gca().invert_xaxis()


plt.xlabel("Effective Temperature (K)")
plt.ylabel("Luminosity (L☉)")
plt.title("Hertzsprung-Russell Diagram")
plt.legend()
plt.grid(alpha=0.3, which='both')

plt.show()

La clave es la relacion radio luminosidad, L=4πR2σT

In [ ]:
T_sun = 5772  # K

Te_iso = 10**iso100["logTe"]
L_iso  = 10**iso100["logL"]
R_iso = np.sqrt(L_iso) * (T_sun / Te_iso)**2
R2 = 0.154

In [ ]:
mask = (L_iso > 1e-8) & (L_iso < 1e2)

Te_iso_clean = Te_iso[mask]
L_iso_clean  = L_iso[mask]
R_iso_clean  = R_iso[mask]


In [ ]:
idx = np.argmin(np.abs(R_iso_clean - R2))

Te_secondary = Te_iso_clean[idx]
L_secondary  = L_iso_clean[idx]

In [ ]:
plt.figure(figsize=(7,6))

# Isochrone limpia
plt.scatter(Te_iso_clean, L_iso_clean, s=8, alpha=0.5, label='PARSEC 100 Myr')

# Primaria (Gaia)
plt.scatter(Teff_star, Lum_star,
            color='crimson', s=140,
            edgecolor='white', linewidth=1.5,
            zorder=5, label='Primary star')

# Secundaria (derivada del radio)
plt.scatter(Te_secondary, L_secondary,
            color='dodgerblue', s=140,
            edgecolor='white', linewidth=1.5,
            zorder=6, label='Secondary star')

plt.xscale('log')
plt.yscale('log')
plt.gca().invert_xaxis()
plt.ylim(1e-3, 1e2)

plt.xlabel("Effective Temperature (K)")
plt.ylabel("Luminosity (L☉)")
plt.title("Hertzsprung-Russell Diagram")
plt.legend()
plt.grid(alpha=0.25, which='both')

plt.tight_layout()
plt.show()


# intentando incertidumbres



In [ ]:

logTe_iso = iso100["logTe"][mask].values
logL_iso  = iso100["logL"][mask].values
R_iso_clean = iso100["Mass"][mask].values   
Te_iso_clean = 10**logTe_iso
L_iso_clean  = 10**logL_iso

In [ ]:
from scipy.interpolate import interp1d
import numpy as np

order = np.argsort(R_iso_clean)

R_sorted    = R_iso_clean[order]
logTe_sorted = logTe_iso[order]
logL_sorted  = logL_iso[order]

Teff_interp = interp1d(R_sorted, 10**logTe_sorted, bounds_error=False, fill_value="extrapolate")
LogL_interp = interp1d(R_sorted, logL_sorted,     bounds_error=False, fill_value="extrapolate")


In [ ]:
Te2 = Teff_interp(R2)

logL2 = LogL_interp(R2)
L2 = 10**logL2


In [ ]:
from scipy.interpolate import interp1d
import numpy as np

# Ordenar por radio (importante para interpolar bien)
order = np.argsort(R_iso_clean)

R_sorted  = R_iso_clean[order]
Te_sorted = Te_iso_clean[order]
L_sorted  = L_iso_clean[order]

Teff_interp = interp1d(R_sorted, Te_sorted, bounds_error=False, fill_value="extrapolate")
Lum_interp  = interp1d(R_sorted, L_sorted,  bounds_error=False, fill_value="extrapolate")


In [ ]:

Te2 = Teff_interp(R2)
L2  = Lum_interp(R2)

sigma_logL2 = sigma_logL_iso
sigma_L2 = L2 * np.log(10) * sigma_logL2

print(f"Secundaria: T = {Te2:.0f} K")
print(f"Secundaria: L = {L2:.4f} ± {sigma_L2:.6f} L☉")

In [ ]:
sigma_logL2 = sigma_logL_iso  # en dex

L_upper = 10**(logL2 + sigma_logL2)
L_lower = 10**(logL2 - sigma_logL2)

yerr_lower = L2 - L_lower
yerr_upper = L_upper - L2

print(L2, L_lower, L_upper)


In [ ]:
plt.figure(figsize=(7,6))

plt.plot(Te_iso_clean, L_iso_clean, alpha=0.3, label='PARSEC 100 Myr')
plt.scatter(Teff_star, Lum_star, color='crimson', s=10, zorder=5, label='Primary')
plt.scatter(Te2, L2, color='navy', s=10, zorder=5, label='Secondary')

plt.errorbar(Te2, L2,
             yerr=[[yerr_lower], [yerr_upper]],
             fmt='none', ecolor='k', capsize=4)

plt.xscale('log')
plt.yscale('log')
plt.gca().invert_xaxis()
plt.ylim(ymin, ymax)

plt.xlabel("Effective Temperature (K)")
plt.ylabel("Luminosity (L☉)")
plt.title("HR Diagram with Isochrone Fit Uncertainty")
plt.legend()
plt.grid(alpha=0.25, which='both')

plt.show()


# intentemos hacer una grid

## Modelo parsec

In [ ]:
path_cat = "/home/luciam/Documents/starry/Resultados_Orion/catalogo_completo_mist_sommers.csv"

cat = pd.read_csv(path_cat)
cat.head


In [ ]:
# ============================================================
# Parámetros MIST y Somers+17% spots
# ============================================================

import numpy as np

# ------------------------------------------------------------
# Seleccionar sistema
# ------------------------------------------------------------

star = cat.iloc[0]
tic_id = "TIC 138829403"
TIC = tic_id

# ============================================================
# MIST (_1)
# ============================================================

teff_MIST = float(star["T_median_1"])
sigma_T_MIST = float(star["T_std_1"])

mass_MIST = float(star["mass_median_1"])
sigma_mass_MIST = float(star["mass_std_1"])

age_MIST = float(star["age_median_1"])
sigma_age_MIST = float(star["age_std_1"])

logL_MIST = float(star["lgL_median_1"])
sigma_logL_MIST = float(star["lgL_std_1"])


# ============================================================
# Somers 2017 + 17% spots (_2)
# ============================================================

teff_SOMERS = float(star["T_median_2"])
sigma_T_SOMERS = float(star["T_std_2"])

mass_SOMERS = float(star["mass_median_2"])
sigma_mass_SOMERS = float(star["mass_std_2"])

age_SOMERS = float(star["age_median_2"])
sigma_age_SOMERS = float(star["age_std_2"])

logL_SOMERS = float(star["lgL_median_2"])
sigma_logL_SOMERS = float(star["lgL_std_2"])


# ============================================================
# Mostrar
# ============================================================

print(f"TIC = {TIC}")
print()
print("MIST")
print("-" * 40)
print(f"Teff       = {teff_MIST:.1f} ± {sigma_T_MIST:.1f} K")
print(f"Masa       = {mass_MIST:.3f} ± {sigma_mass_MIST:.3f} M☉")
print(f"Edad       = {age_MIST:.3f} ± {sigma_age_MIST:.3f} Myr")
print(f"log(L/L☉)  = {logL_MIST:.3f} ± {sigma_logL_MIST:.3f}")

print()
print("Somers 2017 + 17% spots")
print("-" * 40)
print(f"Teff       = {teff_SOMERS:.1f} ± {sigma_T_SOMERS:.1f} K")
print(f"Masa       = {mass_SOMERS:.3f} ± {sigma_mass_SOMERS:.3f} M☉")
print(f"Edad       = {age_SOMERS:.3f} ± {sigma_age_SOMERS:.3f} Myr")
print(f"log(L/L☉)  = {logL_SOMERS:.3f} ± {sigma_logL_SOMERS:.3f}")

In [ ]:
# ============================================================
# 1. Cargar priors
# ============================================================

path_priors = "/home/luciam/Documents/starry/Orion_yceb/priors.csv"
priors = pd.read_csv(path_priors)

# ============================================================
# 2. TIC seleccionado
# ============================================================

tic_id = "TIC 138829403"
tic_num = int(tic_id.replace("TIC ", ""))

# Buscar la estrella
row = priors.loc[priors["TIC"] == tic_num]

if len(row) == 0:
    raise ValueError(f"{tic_id} no fue encontrado en priors.csv")

prior = row.iloc[0]

# ============================================================
# 3. Extraer parámetros
# ============================================================

P_orb = float(prior["P_orb"])

depth_primary = float(prior["Depth_primary"])
depth_primary_err = float(prior["Depth_primary_err"])

depth_secondary = float(prior["Depth_secondary"])
depth_secondary_err = float(prior["Depth_secondary_err"])

duration = float(prior["Duration"])
t0 = float(prior["t0"])

# ============================================================
# 4. Mostrar resultados
# ============================================================

print("=" * 60)
print(f"PRIORS FOTOMÉTRICOS — {tic_id}")
print("=" * 60)

print(f"Periodo orbital       = {P_orb:.6f} d")
print(f"t0                    = {t0:.6f}")
print(f"Duración eclipse      = {duration:.6f} d")

print()
print(f"Profundidad primaria  = {depth_primary:.6f} "
      f"+/- {depth_primary_err:.6f}")

print(f"Profundidad secundaria = {depth_secondary:.6f} "
      f"+/- {depth_secondary_err:.6f}")

print("=" * 60)

In [ ]:
Tsun   = 5772
tic_id = "TIC 35159657"


print(cat.columns.tolist())
r_A = float(cat['rad'][0])   # Radio en R☉
m_A = float(cat['mass'][0])  # Masa en M☉
teff_A = float(cat['Teff'][0])  # Kelvin

teff_err_neg = float(cat['eneg_Teff'][0])  # error inferior en Teff
teff_err_pos = float(cat['epos_Teff'][0])  # error superior en Teff
sigma_T = 0.5 * (teff_err_neg + teff_err_pos)


if 'e_rad' in cat.colnames and np.isfinite(cat['e_rad'][0]):
    sigma_r_A = float(cat['e_rad'][0])
else:
    sigma_r_A = 0.1 * r_A

L_A = r_A**2 * (teff_A/Tsun)**4
sigma_LA = L_A * np.sqrt( (2*sigma_r_A/r_A)**2 + (4*sigma_T/teff_A)**2 )
logL_A = np.log10(L_A)
sigma_logL_A = sigma_LA / (L_A * np.log(10))

print("Luminosidad primaria:", L_A, "L☉")
print("Teff primaria =", teff_A, "+", teff_err_pos, "-", teff_err_neg, "K")
print("Radio primaria:", r_A, "R☉")
print("Masa primaria:", m_A, "M☉")


ages = np.linspace(Age_all.min(), Age_all.max(), 400)
L1_pred = Lum_interp2D(ages, R1)

chi2_1 = ((L1_pred - logL_A)/sigma_logL_A)**2

best_age = ages[np.nanargmin(chi2_1)]
print("Edad por primaria:", 10**best_age/1e6, "Myr")



In [ ]:
if R2 < R_all.min() or R2 > R_all.max():
    print("⚠️ R2 fuera del rango de isocronas")
print("Radio mínimo del modelo:", R_all.min())
print("Radio máximo del modelo:", R_all.max())


## Modelo baraffe

In [ ]:
cols = [
    "Mass", "Teff", "logL", "logg", "Radius", "Li",
    "F33","F33B","F41","F45B","F47","F51","FHa","F57","F63B",
    "F67","F75","F78","F82","F82B","F89","G_RSV","G","G_BP","G_RP"
]

file = "/home/luciam/Documents/catalogos_tesis/iso_barafe/BHAC15_iso.GAIA"

tables = []
rows = []
current_age = None

with open(file) as f:
    for line in f:
        line = line.strip()

        # Detectar nueva edad
        if line.startswith("!  t"):
            # Guardar bloque anterior
            if rows and current_age is not None:
                df = pd.DataFrame(rows, columns=cols)
                df["logAge"] = np.log10(current_age * 1e9)  # Gyr → años
                tables.append(df)
                rows = []

            current_age = float(line.split("=")[1])  # edad en Gyr
            continue

        # Saltar comentarios y encabezados
        if line.startswith("!") or line == "" or line.startswith("BHAC"):
            continue

        # Solo aceptar líneas que empiezan con número (datos)
        if line[0].isdigit():
            values = line.split()
            if len(values) >= len(cols):  # asegurarnos que es fila completa
                rows.append([float(v) for v in values[:len(cols)]])

# Guardar último bloque
if rows and current_age is not None:
    df = pd.DataFrame(rows, columns=cols)
    df["logAge"] = np.log10(current_age * 1e9)
    tables.append(df)

iso_all = pd.concat(tables, ignore_index=True)



In [ ]:
print(iso_all.head())
print("Edades únicas:", np.unique(iso_all.logAge)[:5], "...")
print("Masas:", iso_all.Mass.min(), "–", iso_all.Mass.max())


In [ ]:
from scipy.interpolate import LinearNDInterpolator

points = np.array([iso_all.logAge, iso_all.Mass]).T

interp_R    = LinearNDInterpolator(points, iso_all.Radius)
interp_Teff = LinearNDInterpolator(points, iso_all.Teff)
interp_logL = LinearNDInterpolator(points, iso_all.logL)


In [ ]:
m_grid = np.linspace(0.01, 0.5, 500)
r_model = interp_R(logAge_sys, m_grid)

idx = np.nanargmin(np.abs(r_model - R2))
M2_est = m_grid[idx]

Te2 = interp_Teff(logAge_sys, M2_est)
L2  = 10**interp_logL(logAge_sys, M2_est)

print("Masa secundaria estimada:", M2_est, "Msun")
print("Teff secundaria:", Te2, "K")
print("Luminosidad secundaria:", L2, "Lsun")


In [ ]:
logTe_all = np.log10(Te_all)
logL_all  = np.log10(L_all)

logTe1 = np.log10(teff_A)
logL1  = logL_A        # ya lo tienes

logTe2 = np.log10(Te2)
logL2  = np.log10(L2)
print("Rango logTe isocronas:", logTe_all.min(), logTe_all.max())
print("Rango logL  isocronas:", logL_all.min(), logL_all.max())
print("Primaria:", logTe1, logL1)
sigma_R2 = 0.1 * R2   # 10% típico en sistemas degenerados por contraste


In [ ]:
mask = (
    (L_all > 1e-4) & (L_all < 1e2)   # rango razonable BHAC15
)

Te_plot  = Te_all[mask]
L_plot   = L_all[mask]
Age_plot = Age_all[mask]


In [ ]:
plt.style.use("seaborn-v0_8")
plt.errorbar(
    teff_A, L_A,
    xerr=sigma_T,
    yerr=sigma_LA,
    fmt='o',
    color='red',
    ecolor='darkred',
    elinewidth=1.5,
    capsize=3,
    markersize=8,
    label='Primaria'
)

plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.size'] = 12
plt.rcParams['font.weight'] = 'normal'
plt.rcParams['text.color'] = 'black'
plt.figure(figsize=(7,6))

sc = plt.scatter(
    Te_plot, L_plot,
    c=Age_plot, s=5, cmap='viridis', alpha=0.6
)

plt.scatter(Teff_A, L_A, color='red', s=120, edgecolor='k', label='Primaria')
plt.scatter(Te2, L2, color='cyan', s=120, edgecolor='k', label='Secundaria')

plt.gca().invert_xaxis()  # HR clásico: caliente a la izquierda
plt.yscale('log')

plt.xlabel("Teff (K)")
plt.ylabel("Luminosidad (L☉)")
plt.title("Diagrama HR con isocronas BHAC15")

cbar = plt.colorbar(sc)
cbar.set_label("log Edad (años)")

plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
print("Secundaria:")
print("Teff =", Te2)
print("L =", L2)

# Punto más cercano en el grid
dist = np.sqrt((Te_all-Te2)**2/Te2**2 + (L_all-L2)**2/L2**2)
i = np.argmin(dist)

print("Modelo más cercano:")
print("Teff_model =", Te_all[i])
print("L_model =", L_all[i])
print("Edad_model =", 10**Age_all[i]/1e6, "Myr")


In [ ]:
# Profundidad del tránsito y su error
delta = 0.959      # profundidad medida
sigma_delta = .05   # incertidumbre de la profundidad

# Radio de la primaria (Gaia) y su error
R1 = r_A
sigma_R1 = sigma_r_A

# Radio secundario
R2 = R1 * np.sqrt(delta)

# Error propagado
sigma_R2 = R2 * np.sqrt(
    (sigma_R1 / R1)**2 +
    (0.5 * sigma_delta / delta)**2
)

print("R2 =", R2, "R☉")
print("Error R2 =", sigma_R2, "R☉")


In [ ]:
ages = np.linspace(Age_all.min(), Age_all.max(), 400)
ages = np.asarray(ages)

chi2_min = np.nanmin(chi2_1)
mask_age = chi2_1 < chi2_min + 1

if np.sum(mask_age) > 0:
    age_low  = ages[mask_age].min()
    age_high = ages[mask_age].max()
    sigma_logAge = 0.5 * (age_high - age_low)
else:
    sigma_logAge = 0.05  # fallback razonable
    print("⚠️ No se pudo estimar error de edad, usando valor por defecto")
